## Key Findings from SQL Analysis

- **Office Supplies leads total sales**, followed by Furniture and Technology — consistent with the Task 1 EDA findings.
- **Standard Class is the dominant shipping method**, accounting for the majority of orders (5,186) and total sales value (~$483K), far ahead of Second Class, First Class, and Same Day combined.
- **Sales have grown steadily year over year**, rising from ~$155.7K in 2015 to ~$269.7K in 2018 — a ~73% increase over four years, confirming the upward business trend identified earlier.
- **[Fill in from your Q3 output]** — e.g. "The West/East regions generate the highest sales, while [region] lags

In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv('../data/cleaned_superstore.csv')
conn = sqlite3.connect('../data/superstore.db')
df.to_sql('orders', conn, if_exists='replace', index=False)

print("Data loaded into SQLite successfully!")
print(f"Rows: {len(df)}, Columns: {list(df.columns)}")

Data loaded into SQLite successfully!
Rows: 8655, Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales']


In [2]:
for col in df.columns:
    print(col)

Row ID
Order ID
Order Date
Ship Date
Ship Mode
Customer ID
Customer Name
Segment
Country
City
State
Postal Code
Region
Product ID
Category
Sub-Category
Product Name
Sales


In [3]:
# Basic SQL practice queries

# SELECT + WHERE + ORDER BY + LIMIT
q1 = pd.read_sql("""
    SELECT "Order ID", "Customer Name", "Category", "Sales"
    FROM orders
    WHERE "Sales" > 500
    ORDER BY "Sales" DESC
    LIMIT 10
""", conn)
print(q1)

         Order ID  Customer Name   Category   Sales
0  CA-2015-114321  Nick Crebassa  Furniture  500.24


In [4]:
# GROUP BY + HAVING
q2 = pd.read_sql("""
    SELECT "Category", SUM("Sales") AS total_sales
    FROM orders
    GROUP BY "Category"
    HAVING SUM("Sales") > 10000
    ORDER BY total_sales DESC
""", conn)
print(q2)

          Category  total_sales
0  Office Supplies  337978.7500
1        Furniture  243370.4348
2       Technology  225412.1850


In [5]:
# CTE + Window functions
q3 = pd.read_sql("""
    WITH ranked_sales AS (
        SELECT "Category", "Sub-Category", "Sales",
               RANK() OVER (PARTITION BY "Category" ORDER BY "Sales" DESC) AS sales_rank
        FROM orders
    )
    SELECT * FROM ranked_sales WHERE sales_rank <= 3
""", conn)
print(q3)

          Category Sub-Category    Sales  sales_rank
0        Furniture       Chairs  500.240           1
1        Furniture       Chairs  498.260           2
2        Furniture       Tables  493.920           3
3  Office Supplies   Appliances  499.584           1
4  Office Supplies      Binders  497.940           2
5  Office Supplies      Storage  497.610           3
6       Technology       Phones  499.990           1
7       Technology  Accessories  499.980           2
8       Technology  Accessories  499.950           3


In [6]:
# JOIN example (self-context: customer order count)
q4 = pd.read_sql("""
    SELECT "Customer Name", COUNT("Order ID") AS num_orders, SUM("Sales") AS total_spend
    FROM orders
    GROUP BY "Customer Name"
    ORDER BY total_spend DESC
    LIMIT 10
""", conn)
print(q4)

     Customer Name  num_orders  total_spend
0       Paul Prost          32     4196.928
1  Laura Armstrong          23     3751.798
2     Ken Lonsdale          26     3371.691
3         John Lee          29     3114.772
4    William Brown          31     3101.582
5      Rick Wilson          20     2879.891
6     Noel Staavos          23     2857.753
7    Arthur Gainer          19     2806.851
8      Clay Ludtke          22     2801.042
9       Joel Eaton          20     2783.489


In [7]:
conn.execute("""
CREATE VIEW IF NOT EXISTS monthly_sales AS
SELECT strftime('%Y-%m', "Order Date") AS month, SUM("Sales") AS total_sales
FROM orders
GROUP BY month
""")
print(pd.read_sql("SELECT * FROM monthly_sales ORDER BY month", conn))

      month  total_sales
0   2015-01    5406.9670
1   2015-02    3263.6720
2   2015-03   11967.3240
3   2015-04   11032.7010
4   2015-05   10452.7310
5   2015-06   10238.1566
6   2015-07   10859.4270
7   2015-08   12662.9305
8   2015-09   20542.6848
9   2015-10   13696.0970
10  2015-11   22213.4837
11  2015-12   23391.4655
12  2016-01    5486.9160
13  2016-02    5063.5290
14  2016-03    9952.0440
15  2016-04   13127.2505
16  2016-05   10635.2580
17  2016-06   10161.2990
18  2016-07   12858.6910
19  2016-08   12472.8222
20  2016-09   20190.5780
21  2016-10   15004.3020
22  2016-11   27443.5283
23  2016-12   27095.6280
24  2017-01    8508.9570
25  2017-02    9152.2500
26  2017-03   15139.9460
27  2017-04   12706.4200
28  2017-05   16039.6300
29  2017-06   17741.5220
30  2017-07   15321.1920
31  2017-08   13239.3678
32  2017-09   27815.8195
33  2017-10   15272.4930
34  2017-11   30778.7640
35  2017-12   30129.3720
36  2018-01   11874.9630
37  2018-02   10090.2834
38  2018-03   21955.5408


In [8]:
from sqlalchemy import create_engine

engine = create_engine('sqlite:///../data/superstore.db')
print("SQLAlchemy engine connected!")

SQLAlchemy engine connected!


In [9]:
# 1. Top 5 products by sales
q1 = pd.read_sql("""
    SELECT "Product Name", SUM("Sales") AS total_sales
    FROM orders GROUP BY "Product Name"
    ORDER BY total_sales DESC LIMIT 5
""", engine)
print(q1)

                                       Product Name  total_sales
0                        KI Adjustable-Height Table     3950.781
1   Global Wood Trimmed Manager's Task Chair, Khaki     3621.004
2        Situations Contoured Folding Chairs, 4/Set     2959.866
3         Global High-Back Leather Tilter, Burgundy     2841.069
4  Nortel Meridian M3904 Professional Digital phone     2802.618


In [10]:
# 2. Monthly sales trend
q2 = pd.read_sql("SELECT * FROM monthly_sales ORDER BY month", engine)
print(q2)

      month  total_sales
0   2015-01    5406.9670
1   2015-02    3263.6720
2   2015-03   11967.3240
3   2015-04   11032.7010
4   2015-05   10452.7310
5   2015-06   10238.1566
6   2015-07   10859.4270
7   2015-08   12662.9305
8   2015-09   20542.6848
9   2015-10   13696.0970
10  2015-11   22213.4837
11  2015-12   23391.4655
12  2016-01    5486.9160
13  2016-02    5063.5290
14  2016-03    9952.0440
15  2016-04   13127.2505
16  2016-05   10635.2580
17  2016-06   10161.2990
18  2016-07   12858.6910
19  2016-08   12472.8222
20  2016-09   20190.5780
21  2016-10   15004.3020
22  2016-11   27443.5283
23  2016-12   27095.6280
24  2017-01    8508.9570
25  2017-02    9152.2500
26  2017-03   15139.9460
27  2017-04   12706.4200
28  2017-05   16039.6300
29  2017-06   17741.5220
30  2017-07   15321.1920
31  2017-08   13239.3678
32  2017-09   27815.8195
33  2017-10   15272.4930
34  2017-11   30778.7640
35  2017-12   30129.3720
36  2018-01   11874.9630
37  2018-02   10090.2834
38  2018-03   21955.5408


In [11]:
# 3. Sales by region
q3 = pd.read_sql("""
    SELECT "Region", SUM("Sales") AS total_sales
    FROM orders GROUP BY "Region" ORDER BY total_sales DESC
""", engine)
print(q3)

    Region  total_sales
0     West  269677.1345
1     East  229351.9580
2  Central  180207.3958
3    South  127524.8815


In [12]:
# 4. Top 10 customers by spend
q4 = pd.read_sql("""
    SELECT "Customer Name", SUM("Sales") AS total_spend
    FROM orders GROUP BY "Customer Name"
    ORDER BY total_spend DESC LIMIT 10
""", engine)
print(q4)

     Customer Name  total_spend
0       Paul Prost     4196.928
1  Laura Armstrong     3751.798
2     Ken Lonsdale     3371.691
3         John Lee     3114.772
4    William Brown     3101.582
5      Rick Wilson     2879.891
6     Noel Staavos     2857.753
7    Arthur Gainer     2806.851
8      Clay Ludtke     2801.042
9       Joel Eaton     2783.489


In [13]:
# 5. Sales by category and sub-category
q5 = pd.read_sql("""
    SELECT "Category", "Sub-Category", SUM("Sales") AS total_sales
    FROM orders GROUP BY "Category", "Sub-Category"
    ORDER BY total_sales DESC
""", engine)
print(q5)

           Category Sub-Category  total_sales
0        Technology       Phones  116360.0900
1   Office Supplies      Storage   93655.4840
2        Technology  Accessories   91726.9400
3         Furniture       Chairs   91544.5390
4   Office Supplies        Paper   74207.0540
5         Furniture  Furnishings   67580.5060
6   Office Supplies      Binders   59452.1830
7   Office Supplies   Appliances   46720.7290
8         Furniture       Tables   44075.8235
9         Furniture    Bookcases   40169.5663
10  Office Supplies          Art   25592.3860
11  Office Supplies    Envelopes   15523.3900
12       Technology     Machines   12465.3170
13  Office Supplies       Labels   10932.0620
14  Office Supplies     Supplies    8893.5020
15       Technology      Copiers    4859.8380
16  Office Supplies    Fasteners    3001.9600


In [14]:
# 6. Average order value by segment
q6 = pd.read_sql("""
    SELECT "Segment", AVG("Sales") AS avg_order_value, COUNT(*) AS num_orders
    FROM orders GROUP BY "Segment"
""", engine)
print(q6)

       Segment  avg_order_value  num_orders
0     Consumer        94.906692        4543
1    Corporate        91.563517        2582
2  Home Office        90.969458        1530


In [15]:
# 7. Top 5 states by sales
q7 = pd.read_sql("""
    SELECT "State", SUM("Sales") AS total_sales
    FROM orders GROUP BY "State"
    ORDER BY total_sales DESC LIMIT 5
""", engine)
print(q7)

          State  total_sales
0    California  169805.9015
1      New York   90610.8130
2         Texas   70585.8508
3  Pennsylvania   45393.3630
4    Washington   41204.6160


In [16]:
# 8. Customer segmentation by spend (High/Medium/Low)
q8 = pd.read_sql("""
    WITH customer_spend AS (
        SELECT "Customer Name", SUM("Sales") AS total_spend
        FROM orders GROUP BY "Customer Name"
    )
    SELECT
        CASE
            WHEN total_spend >= 5000 THEN 'High'
            WHEN total_spend >= 1000 THEN 'Medium'
            ELSE 'Low'
        END AS spend_tier,
        COUNT(*) AS num_customers,
        SUM(total_spend) AS tier_total
    FROM customer_spend
    GROUP BY spend_tier
""", engine)
print(q8)

  spend_tier  num_customers   tier_total
0        Low            445  251922.0440
1     Medium            345  554839.3258


In [17]:
# 9. Ship mode preference and average delivery impact on sales
q9 = pd.read_sql("""
    SELECT "Ship Mode", COUNT(*) AS num_orders, SUM("Sales") AS total_sales
    FROM orders GROUP BY "Ship Mode"
    ORDER BY total_sales DESC
""", engine)
print(q9)

        Ship Mode  num_orders  total_sales
0  Standard Class        5186  483387.6107
1    Second Class        1671  158701.6432
2     First Class        1325  121020.4069
3        Same Day         473   43651.7090


In [18]:
# 10. Year-over-year sales growth
q10 = pd.read_sql("""
    SELECT strftime('%Y', "Order Date") AS year, SUM("Sales") AS total_sales
    FROM orders GROUP BY year ORDER BY year
""", engine)
print(q10)

   year  total_sales
0  2015  155727.6401
1  2016  169491.8460
2  2017  211845.7333
3  2018  269696.1504
